In [2]:
!pip install librosa soundfile

In [3]:
import librosa
import soundfile as sf
import numpy as np
import os

In [4]:
SAMPLE_RATE = 8000
DURATION = 3
CLIP_SAMPLES = SAMPLE_RATE * DURATION

def create_audio_clips(source_file, output_folder, n_clips=40):

    os.makedirs(output_folder, exist_ok=True)

    print(f"\nLoading: {source_file}")

    # Load audio
    audio, sr = librosa.load(
        source_file,
        sr=SAMPLE_RATE,
        mono=True
    )

    print(f"Audio length: {len(audio)/sr:.2f} seconds")

    # Normalize
    audio = audio / (np.max(np.abs(audio)) + 1e-9)

    for i in range(n_clips):

        start = i * CLIP_SAMPLES
        end = start + CLIP_SAMPLES

        # If audio is shorter than needed
        if end > len(audio):

            start = np.random.randint(
                0,
                max(1, len(audio) - CLIP_SAMPLES)
            )

            end = start + CLIP_SAMPLES

        clip = audio[start:end]

        output_path = f"{output_folder}/clip_{i:03d}.wav"

        sf.write(
            output_path,
            clip,
            SAMPLE_RATE
        )

    print(f"Saved {n_clips} clips to {output_folder}")

In [12]:
create_audio_clips(
    source_file="raw_siphon.wav.wav",
    output_folder="data/raw/siphon",
    n_clips=40
)


Loading: raw_siphon.wav.wav
Audio length: 103.10 seconds
Saved 40 clips to data/raw/siphon


In [16]:
create_audio_clips(
    source_file="raw_pump_hum.wav",
    output_folder="data/raw/pump_hum",
    n_clips=40
)


Loading: raw_pump_hum.wav
Audio length: 13.31 seconds
Saved 40 clips to data/raw/pump_hum


In [23]:
create_audio_clips(
    source_file="raw_slosh.wav",
    output_folder="data/raw/slosh",
    n_clips=40
)


Loading: raw_slosh.wav
Audio length: 25.81 seconds
Saved 40 clips to data/raw/slosh


In [18]:
os.listdir("data/raw/pump_hum")[:5]

['clip_000.wav',
 'clip_001.wav',
 'clip_002.wav',
 'clip_003.wav',
 'clip_004.wav']

In [19]:
import os

os.listdir("data/raw")

['pump_hum', 'siphon', 'slosh']

In [20]:
!pip install librosa pandas scipy scikit-learn

In [24]:
!python 02_extract_features.py


 BodaShield — Feature Extraction

  Processing 40 clips for 'pump_hum' (label=0)
  Processing 40 clips for 'siphon' (label=2)
  Processing 40 clips for 'slosh' (label=1)

 Saved 120 rows to data\features.csv
 Saved feature params to data\feature_params.json

 Give feature_params.json to the backend person.


In [26]:
!python 03_train_model.py


 BodaShield — Model Training

  Loaded 120 samples, 3 classes

  Label distribution:
label
0    40
2    40
1    40


  FUEL GUARD MODEL (pump/slosh/siphon)
  Test accuracy : 100.0%
  CV mean       : 100.0% ± 0.0%
  Training size : 96
  Test size     : 24

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00         8
           2       1.00      1.00      1.00         8

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24

Confusion matrix:
[[8 0 0]
 [0 8 0]
 [0 0 8]]

 Saved  models/fuel_model.pkl
  No data found for ENGINE KNOCK MODEL (healthy/knock) labels [3, 4] — skipping

  Full report to models\training_report.txt

  Next step: run 04_live_inference.py


In [27]:
!pip install sounddevice pyserial requests matplotlib librosa scipy

In [28]:
!python 04_live_inference.py --file raw_siphon.wav --no-serial


  BodaShield — Live Inference Engine
   Loaded fuel model
  models/engine_model.pkl not found — run 03_train_model.py first
   Running without serial (--no-serial or no --port given)

 Listening... (Ctrl+C to stop)

Figure(1200x700)


C:\Users\MercyKangethe\Downloads\04_live_inference.py:374: UserWarning: frames=None which we can infer the length of, did not pass an explicit *save_count* and passed cache_frame_data=True.  To avoid a possibly unbounded cache, frame data caching has been disabled. To suppress this warning either pass `cache_frame_data=False` or `save_count=MAX_FRAMES`.
  ani = animation.FuncAnimation(fig, update_fn, interval=100, blit=False)
C:\Users\MercyKangethe\anaconda3\Lib\site-packages\matplotlib\animation.py:908: UserWarning: Animation was deleted without rendering anything. This is most likely not intended. To prevent deletion, assign the Animation to a variable, e.g. `anim`, that exists until you output the Animation using `plt.show()` or `anim.save()`.
  warnings.warn(
